# Final workstation run

Every configuration below is copied verbatim from the notebook that produced the number the
paper reports. Nothing is inferred.

| Source | Setting |
|---|---|
| nb 12 | `EPOCHS=200, PATIENCE=25, BATCH_SIZE=2048, DEVICE="cuda:0"` |
| nb 12 | `BEST = dict(hidden_dim=128, n_layers=2, mpn=10, lr=5e-4, loss="focal")` |
| nb 12 | `ModelConfig(hidden_dim=128, n_heads=4, n_layers=2, ffn_mult=2, dropout=0.2, cat_emb_dim=16, use_max_agg=False)` |
| nb 12 | `TrainConfig(..., use_swa=False, patience=25)` |
| nb 11 | `DUMMY = "__none__"`, arm A is `entity_cols=("uid", DUMMY)` |

**Two errors in the previous session, both now fixed.**

1. `BASE_TCFG = TrainConfig()` was never replaced, so both runs used `lr=1e-3`, `batch_size=8192`
   and `use_swa=True` instead of `5e-4`, `2048` and `False`. Every fold trained the full 200
   epochs because early stopping is gated on `not use_swa`.
2. Cell B used `entity_cols=("uid",)`. Notebook 11 uses `("uid", "__none__")`. Since
   `edge_dim = 3 + len(entity_cols)`, the first gives `edge_dim=4` and a different `edge_bias`
   layer, so the architecture changed and not just the topology.

Both previous outputs are deleted automatically below.

| Cell | Job | Cost |
|---|---|---|
| 0-2 | Setup, data, housekeeping | ~2 min |
| 3 | **B** cardholder-only at 200 epochs | ~6 h |
| 4 | **C** converged run, keeping scores | ~9 h |
| 5 | Budget sweep for §5.5 | ~1 min |
| 6 | All nine figures | ~1 min |
| 7 | Verify every number in the paper | seconds |

**Prerequisite.** `run_date_gnn.py` must already have the `scores_path` patch. Cell 0 checks.

## 0 — Setup

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, warnings, time, inspect, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import Image, display


def _find(fn, label=""):
    here = Path.cwd().resolve()
    for c in [here, here.parent, here / "fraud_experiment", here.parent / "fraud_experiment",
              here / "date_gnn", here.parent / "date_gnn",
              here.parent / "sparkov_data_prep" / "prepared",
              here / "sparkov_data_prep" / "prepared"]:
        if (c / fn).exists():
            return c
    for base in [here, here.parent, here.parent.parent]:
        h = list(base.glob(f"**/{fn}"))
        if h:
            return h[0].parent
    raise FileNotFoundError(f"{fn} ({label})")


ENGINE_DIR = _find("run_experiment.py", "engine")
GNN_DIR = _find("run_date_gnn.py", "gnn")
PREPARED_DIR = _find("sparkov_clean_v1.csv", "prepared")
for d in (str(ENGINE_DIR), str(GNN_DIR)):
    if d not in sys.path:
        sys.path.insert(0, d)

import torch
import data_io, temporal_folds, evaluation
from experiment_config import ExperimentConfig
from run_experiment import prepare_frame
from temporal_graph import GraphConfig
from model import ModelConfig
from run_date_gnn import TrainConfig, run_date_gnn_fold, subsample_recent, filter_folds

RESULTS_ROOT = PREPARED_DIR.parent / "experiment_results"
FINAL = RESULTS_ROOT / "final"
MATCHED = RESULTS_ROOT / "matched_features"
FAIRNESS = RESULTS_ROOT / "fairness"
CONVERGED = RESULTS_ROOT / "converged"
OUT = RESULTS_ROOT / "reruns"
FIGS = RESULTS_ROOT / "figures"
for d in (OUT, FIGS):
    d.mkdir(parents=True, exist_ok=True)

patched = "scores_path" in inspect.signature(run_date_gnn_fold).parameters
print("gnn dir     :", GNN_DIR)
print("results     :", RESULTS_ROOT)
print("cuda        :", torch.cuda.is_available())
print("scores_path :", "patched" if patched else "NOT PATCHED, cell 4 will fail")
assert patched, "apply the scores_path patch to run_date_gnn.py first"

gnn dir     : /home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sparkov/date_gnn
results     : /home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sparkov/experiment_results
cuda        : True
scores_path : patched


## 1 — Data

Exactly notebook 12's construction. `num` is derived from the `NUMERIC` tuple plus `uid_feats`
rather than hardcoded, so it cannot drift from what the paper reports.

In [2]:
MAX_TRANSACTIONS = 500_000
DEVICE     = "cuda:0"
EPOCHS     = 200
PATIENCE   = 25
BATCH_SIZE = 2048
SEED       = 42
DUMMY      = "__none__"
BEST = dict(hidden_dim=128, n_layers=2, mpn=10, lr=5e-4, loss="focal")
NUMERIC = ("amt", "distance", "age", "city_pop", "hour", "dayofweek")

cfg = ExperimentConfig(
    prepared_dir=PREPARED_DIR, clean_csv_name="sparkov_clean_v1.csv",
    id_col="TransactionID", target_col="is_fraud", time_col="unix_time",
    week_col="week_idx", amount_col="amt", uid_col="uid",
    non_feature_cols=("uid",), min_train_weeks=8,
)

df, manifest, feature_cols, uid_feats = prepare_frame(cfg)
df = subsample_recent(df, cfg.week_col, MAX_TRANSACTIONS).reset_index(drop=True)
num = [c for c in feature_cols if c in NUMERIC] + uid_feats
y = df[cfg.target_col].to_numpy().astype(int)

folds_all = temporal_folds.make_folds(df, cfg)
folds, fold_report = filter_folds(df, folds_all, "is_fraud", 5, 5)
locked_weeks = sorted(pd.read_csv(FINAL / "date_gnn_final.csv").test_week.astype(int))

assert DUMMY not in df.columns, "dummy entity must not be a real column"
assert df["unix_time"].is_monotonic_increasing, "df must stay time-sorted"
assert len(num) == 11, f"expected 11 features, got {len(num)}: {num}"
assert len(folds) == 17 and sorted(int(w) for w, _, _ in folds) == locked_weeks

print("features :", num)
print("folds    :", len(folds), "weeks", locked_weeks[0], "-", locked_weeks[-1])
print("slice    :", df.shape, "| fraud rate", round(df.is_fraud.mean(), 4))


def mcfg():
    """Model config, verbatim from notebook 12."""
    return ModelConfig(hidden_dim=BEST["hidden_dim"], n_heads=4, n_layers=BEST["n_layers"],
                       ffn_mult=2, dropout=0.2, cat_emb_dim=16, use_max_agg=False)


def tcfg():
    """Train config, verbatim from notebook 12."""
    return TrainConfig(epochs=EPOCHS, lr=BEST["lr"], batch_size=BATCH_SIZE,
                       loss=BEST["loss"], use_swa=False, patience=PATIENCE,
                       device=DEVICE, seed=SEED)


def gcfg(entity_cols=("uid", "merchant"), mpn=None):
    return GraphConfig(time_col="unix_time", entity_cols=entity_cols,
                       max_prior_neighbors=mpn or BEST["mpn"])


print("\ntrain:", tcfg())
print("model:", mcfg())

features : ['hour', 'dayofweek', 'amt', 'city_pop', 'distance', 'age', 'uid_prior_count', 'uid_amt_prior_mean', 'uid_amt_zdev', 'uid_velocity_1d', 'uid_velocity_7d']
folds    : 17 weeks 8 - 24
slice    : (500000, 21) | fraud rate 0.0039

train: TrainConfig(epochs=200, lr=0.0005, weight_decay=0.0001, grad_clip=1.0, val_frac=0.15, patience=25, batch_size=2048, infer_batch_size=16384, loss='focal', focal_gamma=2.0, early_stop_metric='ap', use_swa=False, swa_start_frac=0.6, device='cuda:0', seed=42)
model: ModelConfig(hidden_dim=128, n_heads=4, n_layers=2, ffn_mult=2, dropout=0.2, cat_emb_dim=16, edge_dim=2, use_max_agg=False)


## 2 — Housekeeping

Deletes the two contaminated runs from the previous session, renames the sixteen-feature score
files that have caused three path-order collisions in this project, and deprecates the budget
file whose `xgb_*` columns are the sixteen-feature tree.

In [3]:
# delete the two contaminated runs
for name in ("gnn_11_uid_only_converged.csv", "foldckpt_uid_only_converged.csv",
             "gnn_11_converged_v2.csv", "foldckpt_converged_v2.csv",
             "scores_gnn_11_converged.csv"):
    p = OUT / name
    if p.exists():
        p.unlink()
        print("deleted contaminated:", name)

# rename the sixteen-feature protocol scores
for p in (RESULTS_ROOT / "protocol").glob("scores_P*.csv"):
    if not p.stem.endswith("_16feat"):
        p.rename(p.with_name(f"{p.stem}_16feat{p.suffix}"))
        print("renamed:", p.name)

# deprecate the contaminated budget file
bad = RESULTS_ROOT / "budget" / "budget_sensitivity_11feat.csv"
if bad.exists():
    bad.rename(bad.with_name("DEPRECATED_budget_sensitivity_16feat_xgb.csv"))
    print("deprecated: budget_sensitivity_11feat.csv")

# the tuned baseline should already exist from the earlier session
xt = list(Path(RESULTS_ROOT).glob("**/xgb_11_tuned.csv"))
xs = list(Path(RESULTS_ROOT).glob("**/scores_xgb_11_tuned.csv"))
print("\nxgb_11_tuned.csv        :", xt[0].relative_to(RESULTS_ROOT) if xt else "MISSING")
print("scores_xgb_11_tuned.csv :", xs[0].relative_to(RESULTS_ROOT) if xs else "MISSING")
if xt:
    print("  AP", round(pd.read_csv(xt[0]).ap.mean(), 4))

deleted contaminated: gnn_11_uid_only_converged.csv
deleted contaminated: foldckpt_uid_only_converged.csv
deleted contaminated: gnn_11_converged_v2.csv
deleted contaminated: foldckpt_converged_v2.csv
deleted contaminated: scores_gnn_11_converged.csv

xgb_11_tuned.csv        : reruns/xgb_11_tuned.csv
scores_xgb_11_tuned.csv : reruns/scores_xgb_11_tuned.csv
  AP 0.6361


## 3 — B: cardholder-only arm at the converged budget

Notebook 11 ran this arm at 75 epochs. Rerunning at 200 makes it comparable to the converged
model of Table 5, so Table 6 and Table 5 finally share one training budget.

`entity_cols=("uid", DUMMY)` keeps `edge_dim = 5`, so the model is byte-identical to every other
graph arm and only the topology changes. This is the point notebook 07 makes at length.

In [4]:
p_out = OUT / "gnn_11_uid_only_converged.csv"
print("graph:", gcfg(("uid", DUMMY)))

t0 = time.time()
res = run_date_gnn_fold(
    df, num, [], folds,
    gcfg(("uid", DUMMY)),          # dummy keeps edge_dim = 5
    mcfg(),
    tcfg(),
    target_col="is_fraud", time_col="unix_time", alert_rate=cfg.alert_rate,
    evaluation=evaluation, amount_col="amt",
    checkpoint_path=str(OUT / "foldckpt_uid_only_converged.csv"),
)
res["config"] = "gnn_11_uid_only_converged"
res.to_csv(p_out, index=False)

assert set(res.n_features) == {11}
print(f"\nAP {res.ap.mean():.4f}   P@k {res.precision_at_k.mean():.4f}   "
      f"R@k {res.recall_at_k.mean():.4f}   std {res.ap.std():.4f}   ({(time.time()-t0)/3600:.1f}h)")
print(f"at ceiling {(res.epochs_used >= EPOCHS).sum()}/{len(res)}   "
      f"epochs min {res.epochs_used.min()} median {int(res.epochs_used.median())} "
      f"max {res.epochs_used.max()}")
print("\nfor reference, the same arm at 75 epochs gave 0.7523")

graph: GraphConfig(time_col='unix_time', entity_cols=('uid', '__none__'), max_prior_neighbors=10, time_window_seconds=None)
  [date_gnn_v3] week 8: AP=0.7309 P@k=0.7529 (train=126605, test=16859, ep=78, final=best)
  [date_gnn_v3] week 9: AP=0.8594 P@k=0.6585 (train=140935, test=16231, ep=43, final=best)
  [date_gnn_v3] week 10: AP=0.8071 P@k=0.5125 (train=154731, test=15937, ep=55, final=best)
  [date_gnn_v3] week 11: AP=0.8300 P@k=0.9125 (train=168278, test=15967, ep=56, final=best)
  [date_gnn_v3] week 12: AP=0.8278 P@k=0.9375 (train=181849, test=15999, ep=43, final=best)
  [date_gnn_v3] week 13: AP=0.6099 P@k=0.6250 (train=195449, test=15952, ep=29, final=best)
  [date_gnn_v3] week 14: AP=0.7860 P@k=0.8750 (train=209008, test=15972, ep=33, final=best)
  [date_gnn_v3] week 15: AP=0.8259 P@k=0.6875 (train=222584, test=15914, ep=56, final=best)
  [date_gnn_v3] week 16: AP=0.7775 P@k=0.5250 (train=236111, test=15895, ep=54, final=best)
  [date_gnn_v3] week 17: AP=0.7097 P@k=0.6049 (tra

## 4 — C: converged run, keeping per-fold scores

Identical to notebook 12's call apart from `scores_path`. If this does not reproduce AP 0.8331
the configuration still differs and nothing from it should be used.

In [5]:
SCORES = OUT / "scores_gnn_11_converged.csv"
if SCORES.exists():
    SCORES.unlink()

t0 = time.time()
res = run_date_gnn_fold(
    df, num, [], folds,
    gcfg(("uid", "merchant")),
    mcfg(),
    tcfg(),
    target_col="is_fraud", time_col="unix_time", alert_rate=cfg.alert_rate,
    evaluation=evaluation, amount_col="amt",
    checkpoint_path=str(OUT / "foldckpt_converged_v2.csv"),
    scores_path=str(SCORES),
)
res["config"] = "gnn_11_converged_v2"
res.to_csv(OUT / "gnn_11_converged_v2.csv", index=False)

assert set(res.n_features) == {11}
gap = abs(res.ap.mean() - 0.8331)
print(f"\nnotebook 12  0.8331")
print(f"this run     {res.ap.mean():.4f}   gap {gap:.4f}   ({(time.time()-t0)/3600:.1f}h)")
print("MATCH, configuration confirmed" if gap < 0.002
      else "DIVERGENT, do not use these scores until resolved")
print(f"epochs min {res.epochs_used.min()} median {int(res.epochs_used.median())} "
      f"max {res.epochs_used.max()}   at ceiling {(res.epochs_used >= EPOCHS).sum()}/{len(res)}")
print("  (notebook 12 gave min 42, median 76, max 141, ceiling 0/17)")

sc = pd.read_csv(SCORES)
print(f"\nscores  {len(sc):,} rows   {sc.fold.nunique()} folds   {int(sc.y.sum()):,} positives")
assert len(sc) == 338456 and int(sc.y.sum()) == 1276

  [date_gnn_v3] week 8: AP=0.8139 P@k=0.8471 (train=126605, test=16859, ep=58, final=best)
  [date_gnn_v3] week 9: AP=0.8876 P@k=0.7195 (train=140935, test=16231, ep=65, final=best)
  [date_gnn_v3] week 10: AP=0.7933 P@k=0.5250 (train=154731, test=15937, ep=119, final=best)
  [date_gnn_v3] week 11: AP=0.8602 P@k=0.9500 (train=168278, test=15967, ep=131, final=best)
  [date_gnn_v3] week 12: AP=0.9128 P@k=1.0000 (train=181849, test=15999, ep=144, final=best)
  [date_gnn_v3] week 13: AP=0.6374 P@k=0.5875 (train=195449, test=15952, ep=77, final=best)
  [date_gnn_v3] week 14: AP=0.8318 P@k=0.9250 (train=209008, test=15972, ep=65, final=best)
  [date_gnn_v3] week 15: AP=0.8951 P@k=0.7250 (train=222584, test=15914, ep=104, final=best)
  [date_gnn_v3] week 16: AP=0.8714 P@k=0.5875 (train=236111, test=15895, ep=69, final=best)
  [date_gnn_v3] week 17: AP=0.8722 P@k=0.8025 (train=249622, test=16109, ep=121, final=best)
  [date_gnn_v3] week 18: AP=0.9186 P@k=0.6375 (train=263314, test=15931, ep=1

## 5 — Alert-budget sweep for Section 5.5

Pairs the converged model against the tuned tree. Both score files must be in `reruns/`.

In [10]:
need = [OUT / "scores_gnn_11_converged.csv", OUT / "scores_xgb_11_tuned.csv"]
for p in need:
    print(f"{'ok     ' if p.exists() else 'MISSING'} {p.name}")

if all(p.exists() for p in need):
    r = subprocess.run([sys.executable, str(Path(GNN_DIR) / "budget_sweep_11.py"),
                        str(OUT), str(OUT)], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print("\nrun cell 4 first")

ok      scores_gnn_11_converged.csv
ok      scores_xgb_11_tuned.csv
loading
  scores_gnn_11_converged.csv
     338,456 rows  cols=['row_pos', 'score', 'y', 'fold']
    -> 17 folds, weeks 8-24, positives 1,276
  scores_xgb_11_tuned.csv
     338,456 rows  cols=['row_pos', 'score', 'y', 'fold']
    -> 17 folds, weeks 8-24, positives 1,276
  17 common folds: 8-24

 alert_rate  mean_k  gnn_p_at_k  gnn_r_at_k  xgb11_p_at_k  xgb11_r_at_k  delta_p  delta_r  p_precision  p_recall  folds_won_p  n_folds
      0.001      20      0.9398      0.2965        0.8413        0.2589   0.0985   0.0376       0.0032    0.0033           11       17
      0.002      40      0.8973      0.5076        0.7686        0.4317   0.1287   0.0759       0.0004    0.0004           16       17
      0.005     100      0.6509      0.8177        0.5142        0.6530   0.1367   0.1646       0.0004    0.0004           16       17
      0.010     199      0.3633      0.8867        0.3058        0.7564   0.0575   0.1303       0

In [11]:
x = pd.read_csv(OUT/"xgb_11_tuned.csv")
g = pd.read_csv(OUT/"gnn_11_converged_v2.csv")
print(x[x.test_week==22][["ap","precision_at_k"]].to_string(index=False))
print(g[g.test_week==22][["ap","precision_at_k"]].to_string(index=False))
print("xgb per-fold AP range:", round(x.ap.min(),3), "to", round(x.ap.max(),3))

      ap  precision_at_k
0.435466           0.375
      ap  precision_at_k
0.628718           0.475
xgb per-fold AP range: 0.383 to 0.862


## 6 — Figures

In [19]:
SCRIPTS = {
    "fig_method.py": ["F1_protocol.png", "F2_graph.png", "F3_architecture.png"],
    "fig_5_1.py": ["F4_per_fold_matched.png"],
    "fig_5_3.py": ["F5_protocol_ladder.png", "F6_interaction.png"],
    "fig_5_4.py": ["F7_folds_convergence.png", "F8_effect_sizes.png"],
    "fig_5_5.py": ["F9_budget_sensitivity.png"],
}

for script, outputs in SCRIPTS.items():
    path = Path(GNN_DIR) / script
    if not path.exists():
        print("missing script:", script)
        continue
    args = [str(FIGS)] if script == "fig_method.py" else [str(RESULTS_ROOT), str(FIGS)]
    r = subprocess.run([sys.executable, str(path)] + args, capture_output=True, text=True)
    print(f"--- {script}\n{(r.stdout or r.stderr).strip()}")

for outs in SCRIPTS.values():
    for f in outs:
        if (FIGS / f).exists():
            display(Image(filename=str(FIGS / f)))

--- fig_method.py
wrote F1_protocol and F2_graph to /home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sparkov/experiment_results/figures
wrote F1_protocol, F2_graph and F3_architecture to /home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sparkov/experiment_results/figures
--- fig_5_1.py
loading
  reruns/gnn_11_converged.csv                             AP 0.8427  17 folds
  reruns/xgb_11_tuned.csv                                 AP 0.6361  17 folds
  both arms match the manuscript

GNN 0.8427 | XGB 0.6361 | GNN above on 17/17 folds
--- fig_5_3.py
reading /home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sparkov/experiment_results/protocol_11/per_week_all_arms.csv
5 arms, 16 common weeks: 8-23
wrote F5_protocol_ladder and F6_interaction (png + pdf) to /home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sp

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

## 7 — Verify

Every number the manuscript reports, recomputed from whatever is now on disk.

In [8]:
from scipy.stats import wilcoxon

r3 = lambda v: round(v, 3)


def load(*names):
    for n in names:
        for d in (OUT, RESULTS_ROOT):
            hits = list(Path(d).glob(f"**/{n}"))
            if hits:
                return pd.read_csv(hits[0]).sort_values("test_week").reset_index(drop=True)
    return None


g = load("gnn_11_converged_v2.csv", "gnn_11_converged.csv")
x = load("xgb_11_tuned.csv")
u = load("gnn_11_uid_only_converged.csv")
mx = load("maxagg_11.csv")
d3 = load("depth3_11.csv")

for lbl, v in [("converged", g), ("xgb tuned", x), ("uid-only", u),
               ("max-pool", mx), ("depth-3", d3)]:
    print(f"{lbl:12s} {'found' if v is not None else 'MISSING'}")


def cmp(a, b, label):
    print(f"\n{label}")
    for m in ("ap", "precision_at_k", "recall_at_k"):
        d = a[m].values - b[m].values
        print(f"  {m:15s} {a[m].mean():.4f} vs {b[m].mean():.4f}   "
              f"disp {r3(a[m].mean()) - r3(b[m].mean()):+.3f}   "
              f"p {wilcoxon(a[m], b[m])[1]:.5f}   {(d > 0).sum()}/{len(d)}")


if g is not None and x is not None:
    cmp(g, x, "TABLE 5   paper: 0.833 / 0.637  +0.196  p 0.00002  17/17")

if all(v is not None for v in (x, u, g)):
    s = r3(u.ap.mean()) - r3(x.ap.mean())
    mch = r3(g.ap.mean()) - r3(u.ap.mean())
    tot = r3(g.ap.mean()) - r3(x.ap.mean())
    print(f"\nTABLE 6   paper: 0.637->0.752 +0.115, 0.752->0.858 +0.106")
    print(f"  structure {r3(x.ap.mean()):.3f} -> {r3(u.ap.mean()):.3f}  {s:+.3f}  "
          f"p {wilcoxon(u.ap, x.ap)[1]:.5f}  {(u.ap.values > x.ap.values).sum()}/{len(u)}")
    print(f"  merchant  {r3(u.ap.mean()):.3f} -> {r3(g.ap.mean()):.3f}  {mch:+.3f}  "
          f"p {wilcoxon(g.ap, u.ap)[1]:.5f}  {(g.ap.values > u.ap.values).sum()}/{len(g)}")
    print(f"  sum {s + mch:+.3f}   total {tot:+.3f}   "
          f"{'consistent' if abs(s + mch - tot) < 1e-9 else 'RESIDUAL, check rounding'}")
    print(f"  std chain {x.ap.std():.3f} -> {u.ap.std():.3f} -> {g.ap.std():.3f}")

if g is not None and x is not None:
    tg = (g.precision_at_k * g.k).mean()
    tx = (x.precision_at_k * x.k).mean()
    print(f"\nOPERATING POINT   paper: 58 vs 46, twelve additional")
    print(f"  {tg:.1f} vs {tx:.1f}   difference {tg - tx:.1f}")
    m = g.test_week != 24
    print(f"\nWEEK 24 CHECK   paper: +0.201  p 0.00003  16/16")
    print(f"  {r3(g[m].ap.mean()) - r3(x[m].ap.mean()):+.3f}  "
          f"p {wilcoxon(g[m].ap, x[m].ap)[1]:.5f}  "
          f"{(g[m].ap.values > x[m].ap.values).sum()}/{m.sum()}")

if g is not None and mx is not None and d3 is not None:
    print(f"\nARCHITECTURE   paper: max-pool -0.009 p 0.352, depth-3 +0.024 p 0.196")
    for lbl, a in (("max-pool", mx), ("depth-3", d3)):
        print(f"  {lbl:9s} {a.ap.mean():.4f}  {r3(a.ap.mean()) - r3(g.ap.mean()):+.3f}  "
              f"p {wilcoxon(a.ap, g.ap)[1]:.3f}  {(a.ap.values > g.ap.values).sum()}/17")

b = load("budget_sweep_11feat_matched.csv")
print("\nBUDGET SWEEP")
print(b.round(4).to_string(index=False) if b is not None else "  not yet produced")

converged    found
xgb tuned    found
uid-only     found
max-pool     found
depth-3      found

TABLE 5   paper: 0.833 / 0.637  +0.196  p 0.00002  17/17
  ap              0.8427 vs 0.6361   disp +0.207   p 0.00002   17/17
  precision_at_k  0.6493 vs 0.5134   disp +0.136   p 0.00044   16/17
  recall_at_k     0.8186 vs 0.6542   disp +0.165   p 0.00044   16/17

TABLE 6   paper: 0.637->0.752 +0.115, 0.752->0.858 +0.106
  structure 0.636 -> 0.773  +0.137  p 0.00003  16/17
  merchant  0.773 -> 0.843  +0.070  p 0.00168  14/17
  sum +0.207   total +0.207   consistent
  std chain 0.134 -> 0.097 -> 0.098

OPERATING POINT   paper: 58 vs 46, twelve additional
  59.2 vs 46.8   difference 12.5

WEEK 24 CHECK   paper: +0.201  p 0.00003  16/16
  +0.211  p 0.00003  16/16

ARCHITECTURE   paper: max-pool -0.009 p 0.352, depth-3 +0.024 p 0.196
  max-pool  0.8237  -0.019  p 0.030  4/17
  depth-3   0.8568  +0.014  p 0.278  11/17

BUDGET SWEEP
  not yet produced


In [9]:
r = subprocess.run([sys.executable, str(Path(GNN_DIR)/"budget_sweep_11.py"),
                    str(OUT), str(OUT)], capture_output=True, text=True)
print("STDOUT:\n", r.stdout)
print("STDERR:\n", r.stderr)
print("returncode:", r.returncode)

STDOUT:
 loading

STDERR:
 Traceback (most recent call last):
  File "/home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sparkov/date_gnn/budget_sweep_11.py", line 63, in <module>
    G = per_fold(load('scores_P3_rolling.csv', prefer='protocol_11'))
                 ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sparkov/date_gnn/budget_sweep_11.py", line 38, in load
    p = find(fname, prefer); d = pd.read_csv(p)
  File "/home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sparkov/date_gnn/budget_sweep_11.py", line 25, in find
    raise FileNotFoundError(f"{name} not found under {os.path.abspath(ROOT)}")
FileNotFoundError: scores_P3_rolling.csv not found under /home/talal/Desktop/Fraud Experiment/Second Paper/Start Over, Strick approach/Data Preprocessing Sparkov/experiment_results/reruns


In [13]:
for p in sorted(Path(RESULTS_ROOT).rglob("xgb_11_tuned.csv")):
    print(p.relative_to(RESULTS_ROOT), pd.read_csv(p).ap.mean().round(4))
for p in sorted(Path(RESULTS_ROOT).rglob("gnn_11_converged*.csv")):
    print(p.relative_to(RESULTS_ROOT), pd.read_csv(p).ap.mean().round(4))

reruns/xgb_11_tuned.csv 0.6361
converged/gnn_11_converged.csv 0.8331
reruns/gnn_11_converged_v2.csv 0.8427
